# Crop Suitability & Recommendation — Exploratory Analysis

This notebook is the exploratory record behind the project: dataset inspection,
crop-name reconciliation, threshold selection, label construction and model
diagnostics.

The production code lives in `src/crop_suitability/`. This notebook imports it
rather than redefining it, so there is a single source of truth for paths,
thresholds and feature layouts.

Run order: `data` → `train` → `evaluate`, or just run the cells below.


## Setup

In [ ]:
# Run from the repository root so the package is importable.
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from crop_suitability import config, data, features

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")

## 1. Inspect the raw datasets

In [ ]:
df_rec = data.load_crop_recommendation()
df_yield = data.load_crop_yield()

print("recommendation:", df_rec.shape, list(df_rec.columns))
print("yield:        ", df_yield.shape, list(df_yield.columns))
df_rec.head()

In [ ]:
df_yield.head()

In [ ]:
# Which crops does each dataset know about?
print("recommendation crops:", sorted(df_rec[config.CROP_COLUMN].unique()))
print()
print("yield crops:", sorted(df_yield[config.CROP_COLUMN].unique()))

## 2. Reconcile crop names

The two datasets spell crops differently, so names are fuzzy-matched with
`rapidfuzz` and anything scoring below 80 is dropped rather than guessed at.

In [ ]:
mapping = data.build_crop_name_mapping(
    sorted(df_rec[config.CROP_COLUMN].unique()),
    sorted(df_yield[config.CROP_COLUMN].unique()),
)
mapping

In [ ]:
unmatched = set(df_rec[config.CROP_COLUMN].unique()) - set(mapping)
print(f"{len(unmatched)} crops dropped for lack of yield data:")
print(sorted(unmatched))

## 3. Merge

The join key is the crop name alone, so each soil/weather reading is paired with
every yield record for that crop. This is a cross product — note the row count.

In [ ]:
aligned = data.align_crop_names(df_rec, df_yield)
merged = data.merge_datasets(aligned, df_yield)

print("merged shape:", merged.shape)
print()
print(merged[config.CROP_COLUMN].value_counts().to_string())
merged.head()

In [ ]:
# How much of that row count is genuine variation?
distinct = merged[config.BASE_FEATURES].drop_duplicates().shape[0]
print(f"distinct soil/weather vectors: {distinct}")
print(f"rows per distinct vector:      {len(merged) / distinct:.0f}")
print()
print("This duplication is why a random train/test split leaks. See the README.")

## 4. Yield statistics and threshold selection

A row is labelled suitable when its yield clears a per-crop threshold. The
thresholds live in `config.YIELD_THRESHOLDS`; the check below confirms each one
is actually attainable in the data.

In [ ]:
stats = data.yield_statistics(merged)
stats

In [ ]:
for _, row in stats.iterrows():
    threshold = config.YIELD_THRESHOLDS.get(row[config.CROP_COLUMN])
    if threshold is None:
        continue
    status = "ok" if row["max"] >= threshold else "UNREACHABLE"
    print(f"{row[config.CROP_COLUMN]:<10} max={row['max']:>12.2f}  threshold={threshold:>10.1f}  {status}")

## 5. Build the label

In [ ]:
labelled = data.add_suitability_label(merged)

print(labelled[config.SUITABILITY_COLUMN].value_counts().to_string())
print()
print((labelled[config.SUITABILITY_COLUMN].value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
# Per-crop balance: crops at the extremes carry little signal.
per_crop = (
    labelled.groupby(config.CROP_COLUMN)[config.SUITABILITY_COLUMN]
    .value_counts(normalize=True)
    .unstack()
    .fillna(0)
)
per_crop

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=labelled,
    x=config.CROP_COLUMN,
    y=config.YIELD_COLUMN,
    hue=config.SUITABILITY_COLUMN,
    showfliers=False,
    ax=ax,
)
ax.set_yscale("symlog")  # coconut is in nuts/ha, the rest in t/ha
ax.set_title("Yield distribution by crop and suitability label")
ax.set_ylabel("Yield (log scale)")
plt.show()

## 6. Train

`train.py` does the same thing from the command line:

```bash
python -m crop_suitability.train --cv
```

A sample is used here to keep the notebook quick.

In [ ]:
from crop_suitability.train import train_recommendation_model, train_suitability_model

sample = labelled.sample(n=40_000, random_state=config.RANDOM_STATE)

suit_model, suit_metrics = train_suitability_model(sample)
print(f"suitability accuracy: {suit_metrics['accuracy']:.4f}")

In [ ]:
crop_model, crop_metrics = train_recommendation_model(sample)
print(f"recommendation accuracy: {crop_metrics['accuracy']:.4f}")
print()
print("A perfect score here is memorisation of duplicated rows, not skill.")

## 7. Diagnostics

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

from crop_suitability.evaluate import plot_confusion_matrix

X1 = features.build_suitability_features(sample)
y1 = sample[config.SUITABILITY_COLUMN]
_, X1_test, _, y1_test = train_test_split(
    X1, y1, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y1
)

y1_pred = suit_model.predict(X1_test)
print(classification_report(y1_test, y1_pred, zero_division=0))
plot_confusion_matrix(y1_test, y1_pred, labels=[0, 1], title="Suitability")
plt.show()

In [ ]:
# Feature attributions. Needs `pip install shap`.
import shap

explainer = shap.TreeExplainer(suit_model)
subset = X1_test.sample(n=min(2_000, len(X1_test)), random_state=config.RANDOM_STATE)
shap.summary_plot(explainer.shap_values(subset), subset)

## 8. Inference

The `CropAdvisor` class is what the Gradio app and CLI both use.

In [ ]:
from crop_suitability.predict import CropAdvisor, run_self_test

advisor = CropAdvisor()  # loads the committed models from models/
_ = run_self_test(advisor)

In [ ]:
conditions = dict(N=90, P=42, K=43, temperature=25, humidity=80, ph=6.5, rainfall=210)

print(advisor.analyse("rice", **conditions))
print()
for crop, score in advisor.rank_crops(**conditions):
    print(f"{crop:<10} {score:6.1%}")

## 9. Launch the app

```bash
python -m crop_suitability.app
```

Or inline:

In [ ]:
# from crop_suitability.app import build_interface
# build_interface(advisor).launch()